# Worked solutions · Section 4 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F5.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
eq. (1.2.1, `eq.bookFold`), Remark 1.2.5 (`remark.passiveDirection`).
Prices are integer counts of ticks,
and a side of a book is written as its dictionary from price to size, best first, `{1000: 100, 999: 200}`,
whatever the order in which Python would print its keys.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `04-folding-the-stream` runs them and verifies every key.

# Section 4 · The book as a fold

**F1. The fold and its accumulator.**
The book is the accumulator of the fold
$$\mathcal B_k = F(\mathcal B_{k-1}, m_k), \qquad \mathcal B_0 = \varnothing,$$
eq. (1.2.1, `eq.bookFold`): it holds one state, the current one,
and the sequence $(\mathcal B_k)_k$ is produced by whatever drives the fold
(the sentence after the equation).
In the package the accumulator is an `AggregateBook`,
which holds each side as a dictionary from price to size, `bids` and `asks`,
and $F$ is its method `apply`, which changes the book in place (Section 1.2.3 (`sec.foldImplementation`))
and returns `None` when `record` is `False` (`unito26.lob.orderbook.AggregateBook.apply`).
In Python a name, an entry of a list and an argument bound to the book all refer to that one object;
`copy` returns an independent book with the same levels.

**F2. The transition.**
A submission $(t, q, p, d)$ is the submission branch of Algorithm 1.2.1 (`algo.bookFold`),
`submit` in Listing 1.2 (`listing.submit`):
while shares remain, it trades against the best price $\pi$ of the side $-d$ as long as $\pi d \le p d$,
each fill at $\pi$, the price of the resting order,
and it then rests what remains at $p$.
By Proposition 1.1.11 (`prop.decompositionOfLimitOrder`)
this is its market part, of $q_M$ shares, followed by its resting part, of $q - q_M$.
A market order carries a sentinel price, 0 for a sell and `sys.maxsize` for a buy;
its remainder rests at the price of its last fill,
and nowhere when it filled nothing (`resting_price`, Section 1.2.3 (`sec.foldImplementation`)).
A withdrawal of $q$ at $p$ on the side $d$ removes at most the size resting there;
a book built with `strict=True` raises instead when the withdrawal asks for more
(the paragraph on `withdraw`, Section 1.2.3 (`sec.foldImplementation`)),
a `ValueError` (`unito26.lob.orderbook.AggregateBook.withdraw`).
Every write goes through `set_size`, which removes a price whose size reaches zero
(Section 1.2.3 (`sec.foldImplementation`), and Listing 1.1 (`listing.aggregateStorage`)).

**F3. One stream.**
Questions 4.1, 4.5 and 4.6 start from bids `{1000: 100}` and asks `{1002: 120}`,
and apply a limit buy of 50 at 1001, then a market sell of 80.
The buy finds the best ask, 1002, above its limit, and rests: bids `{1001: 50, 1000: 100}`.
The sell takes the 50 at 1001 and 30 of the 100 at 1000:
bids `{1000: 70}`, with 80 shares traded.

**F4. The driver `run`.**
`run`, of `unito26.lob.session`, applies each message with `record=False`
and adds `book.last_traded_size`, the shares the message traded, to `tally["traded"]`;
a withdrawal trades nothing.
It records no configuration (Section 1.2.3 (`sec.foldImplementation`)).
A `Counter` reads 0 at a key it does not hold.

**F5. The LOBSTER files.**
Row $k$ of the orderbook file is the configuration after the event on row $k$ of the message file
(the paragraph after Remark 1.2.5 (`remark.passiveDirection`)).
An execution is recorded as the execution of the resting order:
its direction names the passive side, and the aggressor's direction is its opposite,
Remark 1.2.5 (`remark.passiveDirection`).

## 4.1 · `question.foldByReduceOverApply`

The book and the stream of F3, folded three ways:
`fold_a` loops over the messages,
`fold_b` reduces with `transition`, which applies each message to a copy,
and `fold_c` reduces with `apply` itself.
Each run prints the best bid size of the book returned, then of the book passed in.

**Answer: (C).**
`fold_a` prints `70 70` and `fold_b` prints `70 100`; `fold_c` raises `AttributeError`.

By F3 the two messages leave 70 at 1000, the best bid, in whichever book receives them.
- **`fold_a`** applies each message to `book` in place and returns `book`:
  `final` and `book` are one object, `70 70`.
- **`fold_b`**: `reduce(transition, messages, book)` is `transition(transition(book, m1), m2)`.
  Each call copies its argument and applies the message to the copy.
  `final` is the second copy, with 70 at 1000,
  and `book` is never written and keeps its 100: `70 100`.
  This is the fold of eq. (1.2.1, `eq.bookFold`)
  with a transition that returns a new configuration and leaves its argument as it was.
- **`fold_c`**: the accumulator after the first step is what `book.apply(m1, record=False)` returns,
  `None` (F1).
  The second step evaluates `None.apply(m2, record=False)`: `AttributeError`.

**The other choices.**
- (A) `fold_b` prints `70 70`: takes `book.copy()` for an alias of `book`,
  so that `transition` would change `book`.
- (B) `fold_c` prints `70 70`: takes `apply` to return the book it changes;
  it returns `None` when `record` is `False`.
- (D) `fold_a` prints `70 100`: takes a function to receive a copy of its argument;
  `fold_a` changes the book it is given.

## 4.2 · `question.copyingFoldAgainstInPlaceFold`

`fold_a` and `fold_b` of 4.1, timed on 1000 messages:
limit orders and withdrawals of 10 shares at the five prices nearest each touch,
none of which crosses the spread,
from a book with $\ell = 10$, 100 and 1000 occupied prices on each side.

**Answer: (B).**
`fold_b` is the slower, by a factor that grows with $\ell$ and exceeds ten at $\ell = 1000$.

**The work of a message.**
A withdrawal reads one size and writes at most one.
A submission that does not cross looks up the best price of the opposite side once,
finds that it fails the test $\pi d \le p d$, and writes one size (F2).
A write costs $O(1)$, and the lookup is a scan of the $\ell$ prices of a side
by the builtin `max` or `min`, $O(\ell)$:
Table 1.4 (`tab.fasterBookCosts`).
`fold_a` does this work and nothing else.

**The work of a copy.**
`fold_b` does the same work on a copy,
and `copy` calls the method `set_size` once for each occupied price:
about $2\ell$ calls at every message, whatever prices the message reaches.

**The two costs per price.**
Both folds have a cost that grows with $\ell$, at very different rates per price.
The scan of a side runs inside a builtin, one cheap step per price.
A call of `set_size` is a Python method call, which calls `levels_map` and then writes the dictionary,
a cost per price far above that of a step of `max`.
At $\ell = 10$ the 20 calls of the copy cost a few times the work of the message itself;
at $\ell = 1000$ the 2000 calls of the copy outweigh the scan of 1000 prices inside `max` many times over.
The verification notebook prints a ratio between 5 and 6 at $\ell = 10$ and above 20 at $\ell = 1000$.

**The other choices.**
- (A) About the same at every $\ell$:
  misses that the copy writes every occupied price of the book at every message,
  whatever prices the message reaches.
- (C) Slower by about two at every $\ell$:
  takes the copy to cost about what one message costs, whatever $\ell$;
  it makes $2\ell$ calls of `set_size`.
- (D) The same slowdown for both, near the hundredfold growth of $\ell$:
  counts a step of the scan and a call of `set_size` as one unit of work,
  so that work in proportion to $\ell$ dominates both folds;
  the scan runs inside a builtin, at a cost per price far below that of the call of `set_size`
  that the copy makes for each price.
  From $\ell = 10$ to $\ell = 1000$ the verification notebook prints a slowdown of the order of ten for `fold_a`
  and several times larger for `fold_b`.

## 4.3 · `question.recursiveFoldOverAStream`

The fold as a recursion (`fold_a`) and as a loop (`fold_b`),
on $n$ limit buys of 10 shares at 1000 into an empty book, for $n = 500$ and $n = 5000$.
The limit on the depth of recursion is 1000 frames.

**Answer: (D).**
`fold_a` prints `5000` at $n = 500$ and raises `RecursionError` at $n = 5000$;
`fold_b` prints `5000`, then `50000`.

**What is printed.**
The book has no asks, so no buy trades, and each rests its 10 shares at 1000 (F2):
$n$ orders leave $10n$ at 1000, the best bid.

**The depth.**
`fold_a` applies the first message and returns `fold_a(book, messages[1:])`.
Python does not eliminate a call in tail position:
each call of `fold_a` keeps its frame until the call it makes returns.
A stream of $n$ messages makes $n + 1$ nested calls, the last on the empty list,
so the deepest point holds $n + 1$ frames of `fold_a`
and the frames of the calls that led to the first.
`apply` and the methods it calls, `submit`, `best_price`, `levels_map`, `set_size`,
return before the next call of `fold_a` is made,
and add a few frames at a time, not a few per message.
501 frames and a few more stay below 1000; 5001 do not.
`fold_b` keeps one frame of its own for any $n$.

In [ ]:
500 + 1, 5000 + 1, 10 * 500, 10 * 5000

**The other choices.**
- (A) Both print at both lengths:
  takes the recursive call in tail position to reuse the frame of its caller; Python keeps every frame.
- (B) `fold_a` raises at both lengths:
  counts, at each level of the recursion, the frames of `apply` and of the methods it calls beside that of `fold_a`,
  some five a level and about 2500 at $n = 500$; they return before the next level is entered.
- (C) Both raise at $n = 5000$:
  takes the loop of `fold_b` to need a frame per message; a loop runs in the one frame of its function.

## 4.4 · `question.messagesThatCommuteInTheFold`

Bids `{1000: 100, 999: 200}`, asks `{1002: 120, 1003: 80}`,
and three pairs of messages, each applied in both orders:
(i) a limit buy of 50 at 1001 and a limit sell of 40 at 1001;
(ii) a withdrawal of 30 at 1000 on the bid and a market sell of 80;
(iii) the limit buy of 50 at 1001 and a market sell of 30.
Two tuples of dictionaries are equal when their dictionaries hold the same entries,
in whatever order they were inserted.

**Answer: (A).** Pairs ii and iii print `False`.

**Pair i.**
Buy first: 1001 is below the best ask, 1002, so the buy rests 50 at 1001;
the sell at 1001 then meets it and takes 40, leaving 10 at 1001 on the bid.
Sell first: 1001 is above the best bid, 1000, so the sell rests 40 at 1001 on the ask;
the buy takes those 40, finds 1002 above its limit, and rests its last 10 at 1001 (F2).
Both orders leave bids `{1001: 10, 1000: 100, 999: 200}` and the asks as they were: `True`.

**Pair ii.**
Withdrawal first: it leaves 70 at 1000, and the sale takes those 70 and 10 of the 200 at 999:
bids `{999: 190}`.
Sale first: it takes 80 at 1000 and leaves 20;
the withdrawal removes no more than these (F2), and 999 keeps 200: bids `{999: 200}`.
`False`.

**Pair iii.**
Buy first: the sale takes 30 of the 50 at 1001: bids `{1001: 20, 1000: 100, 999: 200}`.
Sale first: it takes 30 at 1000, and the buy then rests 50 at 1001:
bids `{1001: 50, 1000: 70, 999: 200}`.
`False`.

The fold of eq. (1.2.1, `eq.bookFold`) applies the messages in the order of the stream.
Two messages can fail to commute only when one changes what the other finds; pair i shows that this is not enough.

In [ ]:
pair_i = 50 - 40
pair_ii = 200 - (80 - (100 - 30)), 200
pair_iii = (50 - 30, 100), (50, 100 - 30)
pair_i, pair_ii, pair_iii

**The other choices.**
- (B) All three: takes the 10 shares left in pair i to rest on the side of whichever order arrives second;
  in both orders they are the excess of the buy over the sell, and rest on the bid.
- (C) Pair iii alone: takes pair ii to remove 110 shares from the bid in either order,
  as though the withdrawal could take from 999 what 1000 no longer holds;
  it is addressed to 1000, where only 20 remain after the sale.
- (D) Pairs i and iii: takes an order that meets the opposite side to fill at its best price alone, and the rest of it to be cancelled.
  The buy of pair i, arriving second, then loses its last 10, and pair ii agrees, both orders leaving bids `{999: 200}`.
  Algorithm 1.2.1 (`algo.bookFold`) walks on to the next price, and Proposition 1.1.11 (`prop.decompositionOfLimitOrder`) rests the remainder.

## 4.5 · `question.recorderAlignedWithTheOrderbookFile`

The book and the two messages of F3, then a withdrawal of 70 at 1000 on the bid.
`a` appends the book after each message, `b` a copy of it before, `c` a copy of it after.

**Answer: (B).** `[None, None, None]`, `[1000, 1001, 1000]`, `[1001, 1000, None]`.

**The configurations.**
By F3 the first message leaves bids `{1001: 50, 1000: 100}` and the second bids `{1000: 70}`.
The withdrawal of 70 empties 1000, which `set_size` removes (F2): the third leaves no bid.
The best bids of the opening book and after each of the three messages are 1000, 1001, 1000 and `None`.

**The three lists.**
- `a.append(book)` appends the one book three times (F1):
  every entry is the book as it ends, with no bid, `[None, None, None]`.
- `b` copies the book before each message: the opening book and the books after the first two messages,
  `[1000, 1001, 1000]`, one row behind the file.
- `c` copies it after each message, `[1001, 1000, None]`.

Row $k$ of the orderbook file is the configuration after the event on row $k$ (F5),
so `c` is the list that matches the file row by row.

In [ ]:
100 - (80 - 50), 100 - (80 - 50) - 70

**The other choices.**
- (A) `a` as `[1001, 1000, None]`: takes `a.append(book)` to store the book as it stands;
  it stores a reference to it.
- (C) `b` as `[1001, 1000, None]`: takes the copy made before `apply` for the configuration after the message.
- (D) `[1000, 1000, 1000]`, `[1000, 1001, 1000]`, `[1001, 1000, 1000]`:
  takes the withdrawal to leave 1000 at size 0, in the book and in its copies, so that 1000 stays the best bid;
  every write goes through `set_size`, which removes a price whose size reaches zero,
  after a withdrawal as after a fill, and `copy` writes through it.

## 4.6 · `question.runTwiceOverOneGenerator`

Two books built from the same dictionaries, `{1000: 100}` and `{1002: 120}`,
and the stream of F3 as a generator, passed to `run` once for each book.

**Answer: (C).** `80 0 100`.

- `run(book_a, stream)` loops over the generator to its end.
  The limit buy trades nothing and the market sell 80 (F3), so `first["traded"]` is 80 (F4).
- A generator yields each of its values once.
  The second call of `run` finds it exhausted, folds nothing and returns an empty `Counter`,
  which reads 0 at `"traded"` (F4).
- `from_levels`, of `unito26.lob.orderbook.AggregateBook`,
  writes each level into the dictionaries of the new book through `set_size`,
  so the two books share no storage, with each other or with `bids`.
  `book_b` folds nothing and keeps 100 at 1000.

Section 1.2.4 (`sec.fasterBook`) materialises the stream before two books fold it, for this reason.

**The other choices.**
- (A) `80 80 70`: takes the generator to start again at the second call, as a list would.
- (B) `80 0 70`: takes the two books to share the dictionaries passed to `from_levels`,
  so that the fold of `book_a` would show in `book_b`.
- (D) `KeyError`: takes the `Counter` to raise at a missing key, as a `dict` does.

## 4.7 · `question.spreadsRefoldedFromEmpty`

`spreads_a` folds each prefix `messages[:k]` from the empty book with `run` and reads its spread;
`spreads_b` reads the spread after each message of one fold.
Streams of $n = 250$, 500 and 1000 messages,
on which the book never holds more than ten prices.

**Answer: (D).**
The two lists are equal, and `spreads_a` is the slower by a factor that about doubles with each doubling of $n$.

**The lists.**
`messages[:k]` holds the first $k$ messages, indices 0 to $k - 1$, and ends with the $k$-th.
Its fold from $\mathcal B_0 = \varnothing$ is $\mathcal B_k$, eq. (1.2.1, `eq.bookFold`):
entry $k$ of `spreads_a` is the spread of $\mathcal B_k$, and so is entry $k$ of `spreads_b`.

**The cost.**
`spreads_a` applies $1 + 2 + \dots + n = n(n+1)/2$ messages, 31375, 125250 and 500500, against the $n$ of `spreads_b`.
Each message costs at most a scan of the ten prices of the book, whatever $n$,
so the ratio of the two times grows as $(n+1)/2$, the ratio of the numbers of messages applied: 125.5, 250.5 and 500.5.
`run` records nothing (F4),
and the spread read after each message of `spreads_b` costs two scans of at most five prices.
The verification notebook prints ratios that grow about as $(n+1)/2$.

In [ ]:
250 * 251 // 2, 500 * 501 // 2, 1000 * 1001 // 2, 251 / 2, 501 / 2, 1001 / 2

**The other choices.**
- (A) A factor about the same at every $n$: takes each refold to cost what one message costs;
  the refold of the $k$-th prefix applies $k$ messages.
- (B) `spreads_a` the faster: takes the spread read after every message to cost more than the refolds;
  it reads two best prices of a book of at most ten prices,
  while `spreads_a` applies $n(n+1)/2$ messages.
- (C) The lists differ by one position: reads `messages[:k]` as ending before the $k$-th message,
  so that the first prefix would be empty; for $k = 1$ it holds the first message.

## 4.8 · `question.deltaLogReplayIsIdempotent`

Bids `{1000: 100, 999: 200}`, asks `{1002: 120}`;
a limit buy of 50 at 1000, a market sell of 120, a withdrawal of 40 at 999 on the bid.
The log keeps, for each message, the levels it changed and the new size at each.
The replays: the stream with its first message repeated,
and the deltas with the first repeated, with the first lost, and with the last lost.

**Answer: (A).** `False True True False`.

**The deltas.**
The buy at 1000 is below the best ask, so it rests, and 1000 holds 150.
The market sell takes 120 of them and leaves 30.
The withdrawal leaves 160 at 999.
Each message changes one level, and `deltas` is
(bid, 1000, 150), (bid, 1000, 30), (bid, 999, 160).
The clean replay ends with bids `{1000: 30, 999: 160}`.

In [ ]:
100 + 50, 100 + 50 - 120, 200 - 40, 100 + 50 + 50 - 120

**The replays.**
- The stream with its first message repeated rests 50 shares twice, and 1000 ends at 80: `False`.
- A delta writes an absolute size through `set_size`.
  The first delta delivered twice writes 150 twice, and the next writes 30: `True`.
- The first delta lost: the second writes 30 at 1000 over the opening 100, and the third 160 at 999: `True`.
- The last delta lost: nothing after it writes 999, which keeps its opening 200: `False`.

Section 1.2.3 (`sec.foldImplementation`), on the `DeltaLog`:
a change stored as the new absolute size is idempotent,
so a repeated or a lost one is corrected by the next change at that price.
A lost change is corrected only when a later change at its price follows.

**The other choices.**
- (B) `False False False False`: reads `resting` as a change in size, to be added;
  a signed change needs every record exactly once.
- (C) `False True False False`: sees that a repeat is harmless,
  and misses that the lost first delta is overwritten by the next one at 1000.
- (D) `False True True True`: takes every lost delta to be made good by a later one;
  the last has none at 999.

## 4.9 · `question.recordedStreamFoldedFromEmpty`

A withdrawal of 40 at 1000 on the bid, a market sell of 80 and a limit sell of 30 at 1001,
folded from bids `{1000: 100, 999: 50}`, asks `{1002: 120}`,
from the empty book, and from the empty book built with `strict=True`.

**Answer: (B).** It prints `999 1001` and `None 1001`, then raises `ValueError`.

- **The opening book.** The withdrawal leaves 60 at 1000.
  The market sell takes those 60 and 20 of the 50 at 999, which keeps 30.
  The limit sell at 1001 is above the best bid, 999, so it rests, below the best ask, 1002:
  `999 1001`.
- **The empty book.** The withdrawal finds nothing at 1000 and removes nothing (F2).
  The market sell finds no bid and fills nothing;
  with no last fill its remainder has no price to rest at, `resting_price` returns `None`,
  and it rests nowhere.
  The limit sell rests at 1001: `None 1001`.
- **The strict book.** The withdrawal asks for 40 where nothing rests,
  and the strict book raises `ValueError` before the third line is printed.

A recorded stream begins with messages about orders posted before the recording did
(Section 1.2.3 (`sec.foldImplementation`)),
and a fold from $\mathcal B_0 = \varnothing$, eq. (1.2.1, `eq.bookFold`), cannot reproduce what those orders did.

In [ ]:
100 - 40, 80 - (100 - 40), 50 - (80 - (100 - 40))

**The other choices.**
- (A) The strict book prints `None 1001`:
  misses that `strict=True` refuses a withdrawal of more than rests.
- (C) `None 0` on the second line:
  rests the unfilled market sell at its sentinel price 0, which would become the best ask;
  `resting_price` returns the price of the last fill for a market order, and there is none.
- (D) The empty book raises:
  takes the withdrawal from an empty side to be refused without `strict=True`;
  the default book removes what rests there, here nothing.

## 4.10 · `question.lobsterExecutionsReplayed`

Three executions of visible orders at one time, of 60 and 40 at 1000 and of 50 at 999,
each of a resting buy,
replayed through `run` as withdrawals (`replay_a`), as market orders of direction $-d$ (`replay_b`)
and as market orders of direction $d$ (`replay_c`),
on bids `{1000: 100, 999: 200}` and asks `{1002: 120}`.

**Answer: (C).**
`replay_a` prints `999 150 0`, `replay_b` `999 150 150`, and `replay_c` `1002 30 120`.

The direction column names the resting orders executed, here buys;
the aggressor was a sell, of direction $-d$ (F5).
- **`replay_a`**: the withdrawals of 60 and 40 at 1000 empty it, and the withdrawal of 50 at 999 leaves 150.
  A withdrawal trades nothing (F4): `999 150 0`.
- **`replay_b`**: market sells.
  The first takes 60 at 1000, the second the last 40 there, the third 50 at 999:
  the book of `replay_a`, with 150 shares traded, `999 150 150`.
  The two replays leave the same book because every fill is at the best price on its arrival.
- **`replay_c`**: market buys, against the ask, which holds 120 at 1002.
  The first takes 60 and the second 40, leaving 20;
  the third takes those 20 and finds the ask side empty.
  Its remaining 30 rest at the price of its last fill, 1002, on the bid (F2):
  bids `{1002: 30, 1000: 100, 999: 200}`, with 120 shares traded, `1002 30 120`.
  It reads the column as the direction of the trade,
  which inverts every trade, Remark 1.2.5 (`remark.passiveDirection`).

In [ ]:
100 - 60 - 40, 200 - 50, 60 + 40 + 50, 120 - 60 - 40, 50 - (120 - 60 - 40), 60 + 40 + (120 - 60 - 40)

**The other choices.**
- (A) `replay_a` as `replay_b`: counts the shares withdrawn as traded;
  `run` adds `last_traded_size`, which is zero after a withdrawal.
- (B) `replay_b` and `replay_c` exchanged:
  takes the direction argument of `market_order` to name the side the order trades against,
  as the direction column of the file does; it names the side of the order itself.
- (D) `replay_c` prints `1000 100 120`: takes the 30 unfilled shares of the last market buy to rest nowhere;
  it filled 20 at 1002, and its remainder rests there.

## 4.11 · `question.lobsterHaltRowsReplayed`

A hidden execution of 30 against a resting sell at 1001,
then three halt rows of size 0 whose price column carries the codes $-1$, 0 and 1,
converted by `to_message` and applied to bids `{1000: 100}`, asks `{1002: 120}`.
Each line prints the price column of the row, then the best bid size and the best ask size.

**Answer: (D).** `1001 100 90`, `-1 ValueError`, `0 100 90`, `1 100 90`.

- **Row 1**, of type 5: `to_message` returns `market_order(t, 30, -d)` with $d = -1$,
  a market buy of 30 (F5).
  It takes 30 at 1002, the best ask, a visible level the hidden fill never touched: `1001 100 90`.
- **Row 2**, of type 7, falls through to `limit_order(t, 0, -1, SELL)`.
  `Message.__post_init__` refuses the price $-1$ when the message is constructed,
  inside the `try`: `-1 ValueError`.
- **Rows 3 and 4**: the prices 0 and 1 pass the check, which refuses only a negative price.
  A submission of size 0 enters no pass of the matching loop and has no remainder to rest (F2),
  so the book is unchanged: `0 100 90` and `1 100 90`.

Neither a hidden execution nor a halt moves a visible level,
so on their rows the orderbook file, which holds the configuration after each event (F5),
repeats the configuration before them.
The conversion moves a visible level on the first row.

**The other choices.**
- (A) `1001 70 120`, and 70 on the later lines:
  takes the market buy to trade against the bid,
  as though the direction argument of `market_order` named the side it trades against,
  as the column of the file does.
- (B) `-1 100 90` on the second line: misses that `Message` refuses a negative price.
- (C) `0 ValueError` on the third line: takes the check to refuse 0 as well;
  it refuses only negative prices, and 0 is the price `market_order` gives a sell.